In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.stattools import durbin_watson
import warnings
warnings.filterwarnings('ignore')

from src.config import PROCESSED_DIR, REPORTS_DIR, PSBB_PERIODS

# Load master
df = pd.read_parquet(PROCESSED_DIR / 'jakarta_master.parquet')
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date')

# Focus 2019-2021 untuk COVID analysis
covid_df = df['2019':'2021'].copy()

print(f'COVID analysis dataset: {covid_df.shape}')
print(f'Date range: {covid_df.index.min().date()} to {covid_df.index.max().date()}')
print(f'Valid PM2.5 days: {covid_df["pm25_mean"].notna().sum()}')
print(f'PSBB days: {covid_df["is_psbb"].sum()}')

COVID analysis dataset: (1096, 26)
Date range: 2019-01-01 to 2021-12-31
Valid PM2.5 days: 919
PSBB days: 185


In [2]:
# Interrupted Time Series setup
# Reference point: PSBB start = 2020-04-10

PSBB_START      = pd.Timestamp('2020-04-10')
PSBB_END_STRICT = pd.Timestamp('2020-06-04')
PSBB_END_ALL    = pd.Timestamp('2020-10-11')

# Create ITS variables
covid_df = covid_df.reset_index()

# T = time counter dari awal series
covid_df['T'] = (covid_df['date'] - covid_df['date'].min()).dt.days

# D = treatment indicator (1 = setelah PSBB start)
covid_df['D'] = (covid_df['date'] >= PSBB_START).astype(int)

# P = time since PSBB start (0 before, count after)
covid_df['P'] = np.where(
    covid_df['date'] >= PSBB_START,
    (covid_df['date'] - PSBB_START).dt.days,
    0
)

# Log transform PM2.5 (right-skewed distribution)
covid_df['log_pm25'] = np.log(covid_df['pm25_mean'])

# Month for seasonal control
covid_df['month'] = covid_df['date'].dt.month

print('ITS variables created:')
print(covid_df[['date', 'T', 'D', 'P', 'pm25_mean', 'log_pm25']].head(10).to_string())
print(f'\nPSBB period records: {covid_df["D"].sum()}')

ITS variables created:
        date  T  D  P  pm25_mean  log_pm25
0 2019-01-01  0  0  0        NaN       NaN
1 2019-01-02  1  0  0   5.956522  1.784487
2 2019-01-03  2  0  0   7.260870  1.982500
3 2019-01-04  3  0  0  14.041667  2.642029
4 2019-01-05  4  0  0  26.083333  3.261297
5 2019-01-06  5  0  0  39.333333  3.672072
6 2019-01-07  6  0  0  31.916667  3.463128
7 2019-01-08  7  0  0  25.791667  3.250051
8 2019-01-09  8  0  0  29.166667  3.373027
9 2019-01-10  9  0  0  20.291667  3.010210

PSBB period records: 631


In [3]:
# Define periods
pre_psbb    = covid_df[covid_df['date'] < PSBB_START]
during_psbb = covid_df[
    (covid_df['date'] >= PSBB_START) &
    (covid_df['date'] <= PSBB_END_STRICT)
]
post_psbb = covid_df[covid_df['date'] > PSBB_END_ALL]

def period_stats(data, label):
    pm = data['pm25_mean'].dropna()
    return {
        'Period'     : label,
        'Days'       : len(pm),
        'Mean PM2.5' : pm.mean().round(2),
        'Median PM2.5': pm.median().round(2),
        'Std'        : pm.std().round(2),
        'Min'        : pm.min().round(2),
        'Max'        : pm.max().round(2),
    }

summary = pd.DataFrame([
    period_stats(pre_psbb,    'Pre-PSBB (Jan 2019 - Apr 2020)'),
    period_stats(during_psbb, 'During PSBB Strict (Apr-Jun 2020)'),
    period_stats(post_psbb,   'Post-PSBB (Oct 2020 - Dec 2021)'),
])

print('=== PM2.5 Before/During/After PSBB ===')
print(summary.to_string(index=False))

# Naive reduction
pre_mean    = pre_psbb['pm25_mean'].mean()
during_mean = during_psbb['pm25_mean'].mean()
naive_reduction = (pre_mean - during_mean) / pre_mean * 100
print(f'\nNaive reduction (pre vs during): {naive_reduction:.1f}%')
print('WARNING: This is UNADJUSTED — weather confounders not yet controlled')

=== PM2.5 Before/During/After PSBB ===
                           Period  Days  Mean PM2.5  Median PM2.5   Std   Min    Max
   Pre-PSBB (Jan 2019 - Apr 2020)   418       39.75         37.44 30.50  5.96 426.42
During PSBB Strict (Apr-Jun 2020)    50       39.78         34.71 26.55 19.62 196.00
  Post-PSBB (Oct 2020 - Dec 2021)   356       36.98         32.75 43.89  1.00 469.71

Naive reduction (pre vs during): -0.1%


In [4]:
# ITS Model:
# log(PM2.5) = b0 + b1*T + b2*D + b3*P + g*weather + d*month + e
#
# b1 = baseline trend (pre-PSBB slope)
# b2 = immediate level change at PSBB
# b3 = slope change during PSBB
# g  = weather controls
# d  = seasonal (month) controls

# Prepare model data
model_data = covid_df.dropna(
    subset=['log_pm25', 'temp_mean', 'precipitation', 'windspeed_max', 'humidity_mean']
).copy()

# Month dummies — cast to int so patsy treats as numeric
dummies = pd.get_dummies(model_data['month'], prefix='m', drop_first=True, dtype=int)
model_data = pd.concat([model_data, dummies], axis=1)

month_cols = [c for c in model_data.columns if c.startswith('m_')]

# Build formula
weather_terms = 'temp_mean + precipitation + windspeed_max + humidity_mean'
month_terms   = ' + '.join(month_cols)
formula = f'log_pm25 ~ T + D + P + {weather_terms} + {month_terms}'

# Fit OLS with HAC standard errors (Newey-West, 14-day lag)
model = smf.ols(formula=formula, data=model_data).fit(
    cov_type='HAC',
    cov_kwds={'maxlags': 14}
)

print('=== ITS Regression Results ===')
print(f'N observations : {model.nobs:.0f}')
print(f'R-squared      : {model.rsquared:.4f}')
print(f'Adj. R-squared : {model.rsquared_adj:.4f}')

print(f'\n--- Key Coefficients ---')
for var in ['T', 'D', 'P']:
    coef  = model.params[var]
    pval  = model.pvalues[var]
    ci_lo = model.conf_int().loc[var, 0]
    ci_hi = model.conf_int().loc[var, 1]
    sig   = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
    print(f'{var}: coef={coef:.4f}, p={pval:.4f} {sig}, 95%CI [{ci_lo:.4f}, {ci_hi:.4f}]')

print(f'\n--- Weather Controls ---')
for var in ['temp_mean', 'precipitation', 'windspeed_max', 'humidity_mean']:
    coef = model.params[var]
    pval = model.pvalues[var]
    sig  = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
    print(f'{var}: coef={coef:.4f}, p={pval:.4f} {sig}')

dw = durbin_watson(model.resid)
print(f'\nDurbin-Watson: {dw:.4f} (2.0 = no autocorrelation)')

=== ITS Regression Results ===
N observations : 919
R-squared      : 0.3896
Adj. R-squared : 0.3774

--- Key Coefficients ---
T: coef=0.0005, p=0.0148 *, 95%CI [0.0001, 0.0010]
D: coef=-0.1906, p=0.0498 *, 95%CI [-0.3811, -0.0002]
P: coef=-0.0005, p=0.0781 ns, 95%CI [-0.0010, 0.0001]

--- Weather Controls ---
temp_mean: coef=0.0892, p=0.0333 *
precipitation: coef=-0.0019, p=0.3752 ns
windspeed_max: coef=-0.0902, p=0.0000 ***
humidity_mean: coef=-0.0230, p=0.0028 **

Durbin-Watson: 1.3971 (2.0 = no autocorrelation)


In [5]:
# Convert log coefficients to percentage change
D_coef  = model.params['D']
D_ci_lo = model.conf_int().loc['D', 0]
D_ci_hi = model.conf_int().loc['D', 1]

level_change_pct = (np.exp(D_coef) - 1) * 100
ci_lo_pct        = (np.exp(D_ci_lo) - 1) * 100
ci_hi_pct        = (np.exp(D_ci_hi) - 1) * 100

P_coef           = model.params['P']
slope_change_pct = (np.exp(P_coef) - 1) * 100

print('=== PSBB Effect on PM2.5 (Weather-Adjusted) ===')
print(f'\nImmediate level change (b2):')
print(f'  {level_change_pct:.1f}% ({ci_lo_pct:.1f}% to {ci_hi_pct:.1f}% 95%CI)')
print(f'  p-value: {model.pvalues["D"]:.4f}')

print(f'\nSlope change per day during PSBB (b3):')
print(f'  {slope_change_pct:.2f}% per day')
print(f'  p-value: {model.pvalues["P"]:.4f}')

print(f'\nBaseline trend pre-PSBB (b1):')
T_pct = (np.exp(model.params['T']) - 1) * 100
print(f'  {T_pct:.3f}% per day')
print(f'  p-value: {model.pvalues["T"]:.4f}')

print(f'\n=== Interpretation ===')
if model.pvalues['D'] < 0.05:
    direction = 'decrease' if level_change_pct < 0 else 'increase'
    print(f'PSBB was associated with a statistically significant {direction}')
    print(f'of {abs(level_change_pct):.1f}% in PM2.5 (95%CI: {ci_lo_pct:.1f}% to {ci_hi_pct:.1f}%)')
    print(f'after controlling for weather and seasonality.')
else:
    print(f'No statistically significant immediate level change detected.')
    print(f'Naive reduction of {naive_reduction:.1f}% may be explained by weather/seasonality.')

=== PSBB Effect on PM2.5 (Weather-Adjusted) ===

Immediate level change (b2):
  -17.4% (-31.7% to -0.0% 95%CI)
  p-value: 0.0498

Slope change per day during PSBB (b3):
  -0.05% per day
  p-value: 0.0781

Baseline trend pre-PSBB (b1):
  0.054% per day
  p-value: 0.0148

=== Interpretation ===
PSBB was associated with a statistically significant decrease
of 17.4% in PM2.5 (95%CI: -31.7% to -0.0%)
after controlling for weather and seasonality.


In [6]:
# Generate counterfactual (what would PM2.5 be without PSBB)
model_data_cf        = model_data.copy()
model_data_cf['D']   = 0
model_data_cf['P']   = 0

model_data['predicted']      = np.exp(model.predict(model_data))
model_data['counterfactual'] = np.exp(model.predict(model_data_cf))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=model_data['date'], y=model_data['pm25_mean'],
    mode='lines', name='Observed PM2.5',
    line=dict(color='steelblue', width=1), opacity=0.6,
))
fig.add_trace(go.Scatter(
    x=model_data['date'], y=model_data['predicted'],
    mode='lines', name='Model Fitted',
    line=dict(color='darkblue', width=2),
))
fig.add_trace(go.Scatter(
    x=model_data['date'], y=model_data['counterfactual'],
    mode='lines', name='Counterfactual (no PSBB)',
    line=dict(color='red', width=2, dash='dash'),
))

fig.add_vrect(
    x0='2020-04-10', x1='2020-06-04',
    fillcolor='rgba(255,100,100,0.15)', layer='below', line_width=0,
    annotation_text='PSBB Strict', annotation_position='top left',
)
fig.add_vrect(
    x0='2020-06-05', x1='2020-10-11',
    fillcolor='rgba(255,165,0,0.10)', layer='below', line_width=0,
    annotation_text='PSBB Transition', annotation_position='top left',
)

fig.update_layout(
    title='Jakarta PM2.5: Observed vs Counterfactual (ITS Model)',
    xaxis_title='Date',
    yaxis_title='PM2.5 (µg/m³)',
    template='plotly_white',
    height=500,
    legend=dict(x=0.01, y=0.99),
)

fig.show()
out = str(REPORTS_DIR / 'figures' / '04_covid_its.html')
fig.write_html(out)
print(f'Figure saved: {out}')

Figure saved: C:\AQSEL\me\KARIR\Portofolio Setelah Wisuda\EDA Project Indonesia Urban Air Quality Analysis\reports\figures\04_covid_its.html
